In [1]:
# Install Required Libraries
!pip -q install transformers datasets evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


In [2]:
# Import Libraries
import numpy as np
import torch
import json

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    pipeline
)

import evaluate

In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Define Save Path
SAVE_PATH = "/content/drive/My Drive/NLP/Model"

In [5]:
# Load IMDb Dataset
dataset = load_dataset("imdb")

# Create validation split
splits = dataset["train"].train_test_split(test_size=0.1, seed=42)

train_ds = splits["train"].shuffle(seed=42).select(range(8000))
val_ds   = splits["test"].shuffle(seed=42).select(range(1000))
test_ds  = dataset["test"].shuffle(seed=42).select(range(2000))

print(train_ds)
print(val_ds)
print(test_ds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label'],
    num_rows: 8000
})
Dataset({
    features: ['text', 'label'],
    num_rows: 1000
})
Dataset({
    features: ['text', 'label'],
    num_rows: 2000
})


In [6]:
# Function to print samples
def show_samples(dataset_split, split_name, num_samples=3):
    print(f"\n===== {split_name.upper()} SAMPLES =====")
    for i in range(num_samples):
        print(f"\nSample {i+1}")
        print("Label:", "Positive" if dataset_split[i]["label"] == 1 else "Negative")
        print("Text:", dataset_split[i]["text"][:500], "...")

# Show samples
show_samples(train_ds, "Train")
show_samples(val_ds, "Validation")
show_samples(test_ds, "Test")



===== TRAIN SAMPLES =====

Sample 1
Label: Positive
Text: Most of the Atomic Age monster movies I saw on television as a kid- and some of them, THE BLOB included, scared the daylights outta me. Movies like INVADERS FROM MARS made it all too clear to us "small fry" that kids just weren't to be trusted when it came to things like things invading the Homeworld; THE BLOB just reiterated that fact. I recall, fondly, late evenings spent stretched on the floor watching as Body Snatchers and Martian Invaders and Blobs seeped into an unsuspecting society. There ...

Sample 2
Label: Negative
Text: Normally, I don't like revenge films....PERIOD. But Double Impact has a little more than your typical revenge plot premise. You see, DI has action, accuracy, exotic locales(i.e. Hong Kong), awesome gun battles, and enough martial arts to satisfy your cravings for impressive unarmed combat. Like all revenge flicks, it has villains you'd love to throw out of a plate-glass window on the top floor of a 40

In [7]:
# Load Tokenizer & Tokenize Data
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=256
    )

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
val_tok   = val_ds.map(tokenize, batched=True, remove_columns=["text"])
test_tok  = test_ds.map(tokenize, batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [8]:
# Load Model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
!pip install -q bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.1 MB/s eta 0:00:00


In [11]:
# Load Evaluation Metrics
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")
bertscore = evaluate.load("bertscore")

id2label = {0: "negative", 1: "positive"}

In [12]:
# Define Metric Computation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy.compute(predictions=preds, references=labels)["accuracy"]
    prec = precision.compute(predictions=preds, references=labels, average="binary")["precision"]
    rec = recall.compute(predictions=preds, references=labels, average="binary")["recall"]
    f = f1.compute(predictions=preds, references=labels, average="binary")["f1"]

    # BERTScore on label text
    pred_text = [id2label[p] for p in preds]
    ref_text  = [id2label[l] for l in labels]

    bert = bertscore.compute(
        predictions=pred_text,
        references=ref_text,
        lang="en"
    )

    bert_f1 = float(np.mean(bert["f1"]))

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f,
        "bertscore_f1": bert_f1
    }

In [14]:
import transformers
print(transformers.__version__)

4.57.3


In [15]:
# Training Arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./sentiment_distilbert",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    report_to="none"
)

In [16]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-811800288.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [18]:
# Train the Model
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Bertscore F1
1,0.270200,0.279284,0.898000,0.904297,0.897287,0.900778,0.992390
2,0.156700,0.351992,0.890000,0.904382,0.879845,0.891945,0.991794


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TrainOutput(global_step=1000, training_loss=0.21343083953857422, metrics={'train_runtime': 150.4552, 'train_samples_per_second': 106.344, 'train_steps_per_second': 6.646, 'total_flos': 1288642854125568.0, 'train_loss': 0.21343083953857422, 'epoch': 2.0})

In [19]:
# Evaluate on Validation & Test Sets
val_metrics = trainer.evaluate(val_tok)
test_metrics = trainer.evaluate(test_tok)

print("Validation Metrics:", val_metrics)
print("Test Metrics:", test_metrics)


Validation Metrics: {'eval_loss': 0.27928397059440613, 'eval_accuracy': 0.898, 'eval_precision': 0.904296875, 'eval_recall': 0.8972868217054264, 'eval_f1': 0.9007782101167315, 'eval_bertscore_f1': 0.9923904283046723, 'eval_runtime': 3.0081, 'eval_samples_per_second': 332.44, 'eval_steps_per_second': 10.638, 'epoch': 2.0}
Test Metrics: {'eval_loss': 0.26837456226348877, 'eval_accuracy': 0.8975, 'eval_precision': 0.8825794032723773, 'eval_recall': 0.917, 'eval_f1': 0.8994605198626778, 'eval_bertscore_f1': 0.9923531254529953, 'eval_runtime': 8.3319, 'eval_samples_per_second': 240.04, 'eval_steps_per_second': 7.561, 'epoch': 2.0}


In [20]:
# Save Model & Tokenizer
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model and tokenizer saved to Google Drive.")

Model and tokenizer saved to Google Drive.


In [21]:
# Save Metrics to JSON
metrics = {
    "validation": val_metrics,
    "test": test_metrics
}

with open(f"{SAVE_PATH}/metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print("Metrics saved to Google Drive.")


Metrics saved to Google Drive.


In [22]:
# Inference Demo
classifier = pipeline(
    "text-classification",
    model=trainer.model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

classifier("The movie was absolutely fantastic and very engaging.")
classifier("This was a boring and poorly made film.")


Device set to use cuda:0


[{'label': 'LABEL_0', 'score': 0.9813643097877502}]

In [23]:
# Pretty inference display

examples = [
    "The movie was absolutely fantastic and very engaging.",
    "This was a boring and poorly made film.",
    "The story was interesting, but the acting was disappointing."
]

print("MODEL INFERENCE OUTPUT\n")

for text in examples:
    result = classifier(text)[0]

    label = result["label"]
    score = result["score"]

    print(f"Original : {text}")
    print(f"Predicted: {label} (confidence: {score:.4f})")
    print("-" * 50)

MODEL INFERENCE OUTPUT

Original : The movie was absolutely fantastic and very engaging.
Predicted: LABEL_1 (confidence: 0.9885)
--------------------------------------------------
Original : This was a boring and poorly made film.
Predicted: LABEL_0 (confidence: 0.9814)
--------------------------------------------------
Original : The story was interesting, but the acting was disappointing.
Predicted: LABEL_0 (confidence: 0.9497)
--------------------------------------------------
